# Model Training

Train ML models for price prediction.

In [5]:
import pandas as pd
import sys
# Add project root to path so `src` is importable from the notebook
sys.path.append('..')

from src.utils.config import DEFAULT_TICKERS, RAW_DATA_DIR
from src.modeling.train_model import prepare_features, train_random_forest, save_model
from src.modeling.evaluate_model import evaluate_classification, plot_confusion_matrix, plot_feature_importance

In [6]:
%pip install seaborn --quiet

Note: you may need to restart the kernel to use updated packages.


In [7]:
# Load features
from pathlib import Path
# Use project tickers (DEFAULT_TICKERS) instead of AAPL
ticker = DEFAULT_TICKERS[1]  # change index to pick another project ticker
candidates = [
    Path('../data/indicators') / f'{ticker}_features.csv',
    Path('../../data/indicators') / f'{ticker}_features.csv',
    Path('../../../data/indicators') / f'{ticker}_features.csv',
    Path('../../../../data/indicators') / f'{ticker}_features.csv',
    Path('../../../../data') / 'indicators' / f'{ticker}_features.csv',
 ]
for p in candidates:
    if p.exists():
        data = pd.read_csv(p, index_col=0, parse_dates=True)
        print(f'Loaded features from: {p}')
        break
else:
    raise FileNotFoundError(
        f'Could not find {ticker}_features.csv. Checked paths: {', '.join(str(x) for x in candidates)}'
    )

# Prepare data
X_train, X_test, y_train, y_test, scaler, features = prepare_features(data)

print(f'Training samples: {len(X_train)}')
print(f'Test samples: {len(X_test)}')
print(f'Features: {len(features)}')

Loaded features from: ..\data\indicators\TCS.NS_features.csv
2025-12-17 17:42:12 - src.modeling.train_model - INFO - Prepared data | Train: (862, 39) | Test: (216, 39)
Training samples: 862
Test samples: 216
Features: 39


In [8]:
# Train model
model = train_random_forest(X_train, y_train, n_estimators=100)

2025-12-17 17:42:19 - src.modeling.train_model - INFO - Training Random Forest classifier...
2025-12-17 17:42:19 - src.modeling.train_model - INFO - Random Forest training completed


In [9]:
# Evaluate
y_pred = model.predict(X_test)
metrics = evaluate_classification(y_test, y_pred, model_name='Random Forest')

2025-12-17 17:42:27 - src.modeling.evaluate_model - INFO - 
Random Forest Performance:
2025-12-17 17:42:27 - src.modeling.evaluate_model - INFO - Accuracy: 0.5139
2025-12-17 17:42:27 - src.modeling.evaluate_model - INFO - Precision: 0.6667
2025-12-17 17:42:27 - src.modeling.evaluate_model - INFO - Recall: 0.0909
2025-12-17 17:42:27 - src.modeling.evaluate_model - INFO - F1 Score: 0.1600
2025-12-17 17:42:27 - src.modeling.evaluate_model - INFO - 
Classification Report:
              precision    recall  f1-score   support

           0       0.50      0.95      0.66       106
           1       0.67      0.09      0.16       110

    accuracy                           0.51       216
   macro avg       0.58      0.52      0.41       216
weighted avg       0.59      0.51      0.40       216



In [11]:
# Save model
save_model(model, scaler, features, ticker)
print('Model saved successfully!')

2025-12-17 17:43:22 - src.modeling.train_model - INFO - Saved model artifacts for TCS.NS
Model saved successfully!
